In [1]:
import mlflow
mlflow.set_tracking_uri('http://3.111.55.134:5000')

c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning.")

<Experiment: artifact_location='s3://mlflow-buckets-areeba/10', creation_time=1784308564725, effective_trace_archival_retention=None, experiment_id='10', last_update_time=1784308564725, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning.', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import optuna

In [4]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [5]:
import mlflow.lightgbm  # <-- needed for the LightGBM-native flavor

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model - use the native LightGBM flavor to avoid the
        # skops untrusted-types error (lightgbm.basic.Booster isn't a trusted sklearn type)
        if model_name.lower() == "lightgbm":
            mlflow.lightgbm.log_model(model, f"{model_name}_model")
        else:
            mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
        verbose=-1  # silence LightGBM's per-trial training logs
    )
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for LightGBM, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        random_state=42,
        verbose=-1
    )

    # Log the best model with MLflow, passing the algo_name as "LightGBM"
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test)


# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-18 00:21:33,531] A new study created in memory with name: no-name-799dd611-0db4-41da-9623-db69df049ae3
c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-18 00:21:43,355] Trial 0 finished with value: 0.7433946311562037 and parameters: {'n_estimators': 89, 'learning_rate': 0.047744087007938266, 'max_depth': 8}. Best is trial 0 with value: 0.7433946311562037.
c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-18 00:22:02,522] Trial 1 finished with value: 0.7495244134432467 and parameters: {'n_estimators': 210, 'learning_rate': 0.017878765880797506, 'max_depth': 10}. Best

🏃 View run LightGBM_SMOTE_TFIDF_Trigrams at: http://3.111.55.134:5000/#/experiments/10/runs/fe745063bd304143af9998ca8b6a38b2
🧪 View experiment at: http://3.111.55.134:5000/#/experiments/10
